# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`.

Below, we print the list of RecordSet IDs, their names, and list fields (column names) and their `@id`s.

In [ ]:
# List all available RecordSets and their Fields
record_sets = getattr(metadata, 'record_sets', [])
print("Available RecordSets:")
for rs in record_sets:
    rsid = getattr(rs, '@id', None)
    rsname = getattr(rs, 'name', None)
    print(f"  RecordSet @id: {rsid}, name: {rsname}")
    fields = getattr(rs, 'fields', [])
    print("    Fields (columns):")
    for f in fields:
        fid = getattr(f, '@id', None)
        fname = getattr(f, 'name', None)
        print(f"      Field @id: {fid}, name: {fname}")

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis.

For this dataset, we will extract all RecordSets, but you may specify a particular RecordSet by its `@id`.

In [ ]:
dataframes = {}

# Find all RecordSet @ids
record_set_ids = []
for rs in getattr(metadata, 'record_sets', []):
    rsid = getattr(rs, '@id', None)
    if rsid is not None:
        record_set_ids.append(rsid)

print(f"All record set @ids found: {record_set_ids}")

# For demonstration, extract all record sets, but in this notebook we focus on the first one if available
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Select a record set and one of its numeric fields for demonstration.

In [ ]:
# Example: Use the first RecordSet and a numeric field
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Try to find a numeric field automatically (float, int)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    print(f"Using record set @id: {record_set_id}")
    print(f"First numeric field found: {numeric_field}")

    if numeric_field is not None:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try to find a categorical or grouping field
        group_field = None
        for col in df.columns:
            # Object columns with low cardinality
            if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < max(10, len(df) // 10):
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to explore the FAIR\^2 mass spectrometry protein dataset, load the Croissant schema, identify record sets and their fields by `@id`, filter and normalize data, and visualize distributions. This approach ensures reproducible, schema-driven data science. For more information, consult the repository's documentation or the dataset's original publication.